## Imports 

In [1]:
# PyTorch: tensores, redes neuronales, funciones y optimizadores
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split, Subset
from torchvision.datasets import ImageFolder

# Visión artificial
import torchvision
import torchvision.transforms as transforms

# Cálculo numérico, gráficas y utilidades
import random
from pathlib import Path

# Imports para la configuración del dataset
import random
import shutil
from pathlib import Path

# Cálculo numérico, gráficas y utilidades
import numpy as np
import matplotlib.pyplot as plt
import random
import time

from PIL import Image, ImageFile

## CARGA DEL MODELO ANTERIOR

In [11]:
ckpt = torch.load(MODEL_DIR / "cnn_emociones_ultimo.pt", map_location=DEVICE)

model = CNNBasica(num_classes=len(ckpt["classes"])).to(DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()   # modo evaluación para predecir

print("Clases:", ckpt["classes"])
print(f"Cargado (epoch {ckpt['epoch']}, val acc {ckpt['val_acc']:.1f}%)")

Clases: ['angry', 'happy', 'sad', 'surprised']
Cargado (epoch 22, val acc 37.5%)


## 1. Configuración carpetas dataset

In [ ]:
ORIGEN = Path("./datasets/archive/Emotions-Dataset")  
DESTINO = Path("./datasets/My_dataset")        
IMGS_POR_CLASE = 8000               # nº de imágenes válidas a usar por clase
PROP_TRAIN = 0.8                    # 80% train, 20% val
SEED = 42                        # para que el reparto sea reproducible
MIN_LADO = 32                       # descarta imágenes absurdamente pequeñas
EXTENSIONES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Importante: False para que las imágenes truncadas lancen error al cargarlas
ImageFile.LOAD_TRUNCATED_IMAGES = False

# if DESTINO.exists():
#   shutil.rmtree(DESTINO)

def imagen_valida(ruta: Path) -> bool:
    """True si la imagen se puede abrir y decodificar COMPLETA."""
    try:
        # 1) verify() detecta archivos corruptos en la cabecera/estructura
        with Image.open(ruta) as img:
            img.verify()
        # 2) verify() no detecta truncados: hay que reabrir y cargar los píxeles
        with Image.open(ruta) as img:
            img.load()
            if min(img.size) < MIN_LADO:
                return False
        return True
    except Exception:
        return False


def preparar_clase(carpeta_clase: Path, rng: random.Random):
    candidatas = [p for p in carpeta_clase.iterdir()
                  if p.is_file() and p.suffix.lower() in EXTENSIONES]
    rng.shuffle(candidatas)

    validas, descartadas = [], 0
    for ruta in candidatas:
        if len(validas) >= IMGS_POR_CLASE:
            break
        if imagen_valida(ruta):
            validas.append(ruta)
        else:
            descartadas += 1

    if len(validas) < IMGS_POR_CLASE:
        print(f"  ⚠ Solo se encontraron {len(validas)} imágenes válidas "
              f"(se pedían {IMGS_POR_CLASE})")

    n_train = int(len(validas) * PROP_TRAIN)
    return validas[:n_train], validas[n_train:], descartadas


def copiar(imagenes, carpeta_destino: Path):
    carpeta_destino.mkdir(parents=True, exist_ok=True)
    for ruta in imagenes:
        destino = carpeta_destino / ruta.name
        # evita sobrescribir si hubiera dos archivos con el mismo nombre
        i = 1
        while destino.exists():
            destino = carpeta_destino / f"{ruta.stem}_{i}{ruta.suffix}"
            i += 1
        shutil.copy2(ruta, destino)


def main():
    if not ORIGEN.is_dir():
        raise SystemExit(f"No existe la carpeta de origen: {ORIGEN.resolve()}")

    if DESTINO.exists():
        raise SystemExit(f"{DESTINO} ya existe. Bórrala o cambia DESTINO "
                         "para no mezclar datos antiguos.")

    rng = random.Random(SEED)
    clases = sorted(p for p in ORIGEN.iterdir() if p.is_dir())
    print(f"Clases encontradas: {[c.name for c in clases]}\n")

    for carpeta in clases:
        print(f"Procesando '{carpeta.name}'...")
        train, val, malas = preparar_clase(carpeta, rng)
        copiar(train, DESTINO / "train" / carpeta.name)
        copiar(val, DESTINO / "val" / carpeta.name)
        print(f"  train: {len(train)} | val: {len(val)} | "
              f"descartadas por corruptas/recortadas: {malas}")

    print(f"\nListo. Dataset creado en: {DESTINO.resolve()}")


if __name__ == "__main__":
    main()

SystemExit: No existe la carpeta de origen: C:\Users\casba\OneDrive\universidad\CuartoAño\FSI\CNN\pract2\Emotions-Dataset

## 2. Comprobación existencia de las carpetas train y val dentro de dataset

In [3]:
DATA_DIR = Path("./dataset")  # CAMBIA esta ruta para tu ordenador

if DATA_DIR.exists():
    for split in ("train", "val"):
        folder = DATA_DIR / split
        print(f"\n{split}:")
        if folder.exists():
            for cls in sorted(p for p in folder.iterdir() if p.is_dir()):
                count = sum(1 for f in cls.iterdir() if f.is_file())
                print(f"  {cls.name}: {count} archivos")
        else:
            print("  No existe:", folder)
else:
    print("No se encuentra", DATA_DIR.resolve(), "- crea la estructura o cambia DATA_DIR.")


train:
  angry: 240 archivos
  happy: 240 archivos
  sad: 240 archivos
  surprised: 240 archivos

val:
  angry: 60 archivos
  happy: 60 archivos
  sad: 60 archivos
  surprised: 60 archivos


## 3. preparación de todas las imágenes para que tengan un formato y unas características adecuadas para entrenar el modelo

In [4]:
IMG_SIZE = 128

train_transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])

val_transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])
print("Transformaciones definidas.")

Transformaciones definidas.


## 4. Cargar las carpetas con `ImageFolder`

In [5]:
BATCH_SIZE = 16
EPOCHS = 22
LEARNING_RATE = 1e-3

train_dir, val_dir = DATA_DIR / "train", DATA_DIR / "val"
if train_dir.exists() and val_dir.exists():
    train_ds = ImageFolder(str(train_dir), transform=train_transform)
    val_ds = ImageFolder(str(val_dir), transform=val_transform)
    print("Clases train:", train_ds.classes)
    print("Clases val:", val_ds.classes)
    assert train_ds.classes == val_ds.classes, "Las clases deben coincidir en train y val."
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    xb, yb = next(iter(train_loader))
    print("Forma del lote:", tuple(xb.shape))  # (batch, canales, alto, ancho)
    print("Forma de etiquetas:", tuple(yb.shape))
else:
    print("Crea mi_dataset/train y mi_dataset/val, o modifica DATA_DIR.")

print(train_ds.class_to_idx)

Clases train: ['angry', 'happy', 'sad', 'surprised']
Clases val: ['angry', 'happy', 'sad', 'surprised']
Forma del lote: (16, 3, 128, 128)
Forma de etiquetas: (16,)
{'angry': 0, 'happy': 1, 'sad': 2, 'surprised': 3}


## 5. Semillas y Selección de GPU/CPU

In [6]:
# Semillas para que los experimentos sean reproducibles
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Selección automática de GPU si está disponible; si no, CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Dispositivo:", DEVICE)

PyTorch: 2.15.0+cu132
Torchvision: 0.29.2+cu132
Dispositivo: cuda


## 7. CNN BÁSICA

In [7]:
class CNNBasica(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),       # 16 x 16 x 16
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)        # 32 x 8 x 8
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 32 * 32, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = CNNBasica().to(DEVICE)
print(model)

# Comprobamos el recorrido de una imagen.
with torch.no_grad():
    ejemplo = torch.randn(1, 3, 128, 128).to(DEVICE)
    logits = model(ejemplo)
print("Forma de salida:", logits.shape)
print("Parámetros entrenables:", sum(p.numel() for p in model.parameters() if p.requires_grad))

CNNBasica(
  (features): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=32768, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=4, bias=True)
  )
)
Forma de salida: torch.Size([1, 4])
Parámetros entrenables: 4200036


## 8. Entrenamiento, validación, batches, epochs y almacenamiento del mejor modelo y último

In [10]:
def accuracy_from_logits(logits, labels):
    pred = logits.argmax(dim=1)
    return (pred == labels).sum().item(), labels.size(0)

def run_epoch(model, loader, criterion, optimizer=None, log_batches=False):
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss = 0.0
    total_correct = 0
    total_items = 0
    batch_losses = []

    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for batch_idx, (inputs, labels) in enumerate(loader):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

            if training:
                optimizer.zero_grad()

            logits = model(inputs)
            loss = criterion(logits, labels)

            if training:
                loss.backward()
                optimizer.step()

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            correct, count = accuracy_from_logits(logits, labels)
            total_correct += correct
            total_items += count
            batch_losses.append(loss.item())

    return total_loss / total_items, 100 * total_correct / total_items, batch_losses

MODEL_DIR = Path("./modelos")
MODEL_DIR.mkdir(exist_ok=True)

def train_model(model, train_loader, val_loader, epochs=4, lr=1e-3,
                nombre="cnn_emociones"):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {"train_loss": [], "train_acc": [],
               "val_loss": [], "val_acc": [], "batch_loss": []}
    best_val_acc = 0.0

    for epoch in range(epochs):
        start = time.time()
        train_loss, train_acc, batch_losses = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc, _ = run_epoch(model, val_loader, criterion)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["batch_loss"].append(batch_losses)

        checkpoint = {
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_acc": val_acc,
            "classes": train_loader.dataset.classes,   # orden de las clases
            "history": history,
        }

        # Guarda el último estado siempre
        torch.save(checkpoint, MODEL_DIR / f"{nombre}_ultimo.pt")

        # Guarda el mejor solo si mejora la validación
        marca = ""
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(checkpoint, MODEL_DIR / f"{nombre}_mejor.pt")
            marca = " ← guardado (mejor)"

        print(
            f"Epoch {epoch+1:02d}/{epochs} | "
            f"train loss {train_loss:.3f} acc {train_acc:5.1f}% | "
            f"val loss {val_loss:.3f} acc {val_acc:5.1f}% | "
            f"{time.time()-start:.1f}s{marca}"
        )
    return history

In [12]:
# Descomentar en caso de que se quiera entrenar desde cero (nuevo modelo)
# model = CNNBasica().to(DEVICE) 

history = train_model(
    model,
    train_loader,
    val_loader,
    epochs=EPOCHS,
    lr=LEARNING_RATE
)

Epoch 01/22 | train loss 0.150 acc  95.2% | val loss 2.781 acc  39.6% | 17.2s ← guardado (mejor)
Epoch 02/22 | train loss 0.129 acc  96.4% | val loss 2.932 acc  40.4% | 19.4s ← guardado (mejor)
Epoch 03/22 | train loss 0.146 acc  95.4% | val loss 2.914 acc  39.2% | 16.3s
Epoch 04/22 | train loss 0.137 acc  95.2% | val loss 2.998 acc  35.8% | 16.1s
Epoch 05/22 | train loss 0.092 acc  97.2% | val loss 3.081 acc  36.7% | 16.1s
Epoch 06/22 | train loss 0.140 acc  95.4% | val loss 2.817 acc  38.8% | 15.9s
Epoch 07/22 | train loss 0.077 acc  97.1% | val loss 3.462 acc  33.8% | 16.1s
Epoch 08/22 | train loss 0.100 acc  97.2% | val loss 3.411 acc  37.1% | 16.7s
Epoch 09/22 | train loss 0.099 acc  97.2% | val loss 3.165 acc  39.6% | 16.9s
Epoch 10/22 | train loss 0.090 acc  97.3% | val loss 3.200 acc  41.2% | 17.2s ← guardado (mejor)
Epoch 11/22 | train loss 0.071 acc  97.7% | val loss 3.513 acc  42.5% | 16.4s ← guardado (mejor)
Epoch 12/22 | train loss 0.053 acc  98.1% | val loss 3.611 acc  38

KeyboardInterrupt: 